# 14 — Cross-sub-grid transferability

Apply the native-daily alignment and coverage workflow to Samar–Leyte, Bohol, Cebu, Negros and Panay. Reproduce regional rankings, component surfaces and the selected full-record/Rai alignment panels.

## Key takeaways

Compare the component metrics alongside the score within each sub-grid. The observed best threshold is region-specific. The tested transfer concerns these Visayas sub-grids.

## Run order and exports

Run the cells in order from the repository or notebooks/ folder. Figures use outputs/figures/main/, supplementary/ or diagnostics/. Tables use the matching outputs/tables/ folders. Every static export writes HTML, SVG, PNG and JPEG from the same layout. HTML includes Plotly for offline viewing. JPEG preserves the PNG's pixel dimensions and composites transparency on white.

## 1. Inputs and preprocessing

Load one daily NTL export per region and GHSL mask and the 01:00 series from each NGCP workbook sheet. Align both inputs to the common 2012–2024 calendar. The selection workflow caps direct radiance at P95 within each region and mask.

In [ ]:
from io import BytesIO
from PIL import Image as PILImage
import json
import plotly.io as pio

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "notebooks").is_dir() and (directory / "datasets").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open the kernel from the repository or its notebooks folder. "
        "The repository must contain sibling notebooks/ and datasets/ folders."
    )


ROOT = PROJECT_ROOT
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
FIG_MAIN = OUTPUT_ROOT / "figures" / "main"
FIG_SUPP = OUTPUT_ROOT / "figures" / "supplementary"
FIG_DIAGNOSTIC = OUTPUT_ROOT / "figures" / "diagnostics"
TABLE_MAIN = OUTPUT_ROOT / "tables" / "main"
TABLE_SUPP = OUTPUT_ROOT / "tables" / "supplementary"
TABLE_DIAGNOSTIC = OUTPUT_ROOT / "tables" / "diagnostics"
for folder in (FIG_MAIN, FIG_SUPP, FIG_DIAGNOSTIC, TABLE_MAIN, TABLE_SUPP, TABLE_DIAGNOSTIC):
    folder.mkdir(parents=True, exist_ok=True)

# Match Notebook 04's multi-subgrid analysis period and input locations.
DATE_START = pd.Timestamp("2012-01-19")
DATE_END = pd.Timestamp("2024-12-31")
VALIDATION_DATES = pd.date_range(DATE_START, DATE_END, freq="D", name="date")
NTL_FOLDER = ROOT / "datasets" / "vnp46a2" / "visayas_ghsl_masked_daily_stats"
NGCP_XLSX_PATH = ROOT / "datasets" / "ngcp" / "NGCP_hourly_load.xlsx"

WINDOWS = np.arange(1, 31)
WINDOW_CHECKPOINTS = [1, 4, 7, 14, 21, 30]
SC_GRID = np.arange(0, 101, 1, dtype=float)  # exact search and CSV output
MAIN_TAUS = np.arange(0, 101, 5, dtype=float)  # retain the paper's display interval
SUPP_TAUS = MAIN_TAUS.copy()                   # consistent figures throughout
PRIMARY_REGION = "Samar-Leyte"
EVENTS = [
    ("Haiyan", pd.Timestamp("2013-11-08"), "2013-05-01", "2014-05-31"),
    ("Rai", pd.Timestamp("2021-12-16"), "2021-07-01", "2022-07-31"),
]

GROUPS = [
    ("G8", "10-30", "All Land", "#4E79A7"),
    ("G7", "11-30", "Rural Settlements", "#A8CE74"),
    ("G6", "12-30", "Low–Mid Rural", "#7BAF54"),
    ("G5", "13-30", "Built-Up / Rural Mix", "#375623"),
    ("G4", "21-30", "Urban Fringe", "#D9B900"),
    ("G3", "22-30", "Core & Inner", "#E6A23C"),
    ("G2", "23-30", "Dense Urban", "#A87000"),
    ("G1", "30", "Urban Core", "#D65A4A"),
]
GROUP_LOOKUP = {
    code: {"suffix": suffix, "label": label, "color": color}
    for code, suffix, label, color in GROUPS
}
ORDERED_MASKS = [f"G{i}" for i in range(1, 9)]
MASK_COLORS = {code: GROUP_LOOKUP[code]["color"] for code in ORDERED_MASKS}

REGION_RENAME = {
    "CebuProvince": "Cebu",
    "SamarLeyte": "Samar-Leyte",
    "NegrosIsland": "Negros",
    "PanayIsland": "Panay",
    "BoholProvince": "Bohol",
}
REGION_ORDER = ["Samar-Leyte", "Bohol", "Cebu", "Negros", "Panay"]
GROUP_RENAME = {"30-only": "30"}
SHEET_MAP = {
    "Panay": "PANAY HOURLY LOAD 2013-2024",
    "Negros": "NEGROS HOURLY LOAD 2013-2024",
    "Samar-Leyte": "LEY-SAM HOURLY LOAD 2013-2024",
    "Cebu": "CEBU HOURLY LOAD 2013-2024",
    "Bohol": "BOHOL LOAD 2013-2024",
}

PLOT_TEMPLATE = "plotly_white"
INK = "#233B60"
GRID = "#E8EDF3"
R_CONTRAST_SCALE = "jet_r"
N_CONTRAST_SCALE = "Viridis"
SCORE_CONTRAST_SCALE = "Turbo"
DIVERGING_SCALE = [[0.00, "#B2182B"], [0.50, "#F7F7F7"], [1.00, "#2166AC"]]
POSITIVE_SCALE = "Viridis"
R_COLOUR_POWER = 2.0
N_COLOUR_POWER = 3.0
R_TICKS = np.array([-1.0, -0.5, 0.0, 0.5, 0.75, 1.0])
R_COLOUR_TICKS = np.sign(R_TICKS) * np.abs(R_TICKS) ** R_COLOUR_POWER
N_TICKS = np.array([0, 50, 75, 90, 100], dtype=float)
N_COLOUR_TICKS = (N_TICKS / 100) ** N_COLOUR_POWER

def export_figure(
    fig, output_path, width=None, height=None, scale=3,
    export_html=True, force_date_axis=False, tickformat="%Y", static=True,
):
    """Export one figure layout as HTML, SVG, PNG and JPEG."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure = go.Figure(json.loads(pio.to_json(fig, validate=True)))
    figure.update_layout(
        width=width if width is not None else figure.layout.width or 1200,
        height=height if height is not None else figure.layout.height or 600,
    )
    if force_date_axis:
        figure.update_xaxes(type="date", tickformat=tickformat)

    if static:
        png_bytes = figure.to_image(format="png", scale=scale)
        svg_bytes = figure.to_image(format="svg", scale=1)
        with PILImage.open(BytesIO(png_bytes)) as image:
            rgba = image.convert("RGBA")
            jpeg = PILImage.new("RGB", rgba.size, "white")
            jpeg.paste(rgba, mask=rgba.getchannel("A"))
        buffer = BytesIO()
        jpeg.save(buffer, format="JPEG", quality=95, subsampling=0)
        output_path.with_suffix(".png").write_bytes(png_bytes)
        output_path.with_suffix(".svg").write_bytes(svg_bytes)
        output_path.with_suffix(".jpeg").write_bytes(buffer.getvalue())
    if export_html:
        figure.write_html(
            str(output_path.with_suffix(".html")),
            include_plotlyjs=True, full_html=True,
        )
    return fig
RUN_ADDITIONAL_DIAGNOSTICS = False

In [ ]:
def parse_filename(filepath):
    name = Path(filepath).name.replace("_stats.csv", "")
    parts = name.split("_DNBBRDF_class")
    if len(parts) != 2:
        return "Unknown", "Unknown"
    region, group = parts
    return REGION_RENAME.get(region, region), GROUP_RENAME.get(group, group)


def read_multiregion_ntl(folder):
    csv_files = sorted(Path(folder).glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No NTL CSV files found in {folder.relative_to(PROJECT_ROOT)}")

    frames = []
    for path in csv_files:
        region, group = parse_filename(path)
        if region == "Unknown":
            continue
        try:
            frame = pd.read_csv(path)
        except Exception:
            frame = pd.read_csv(path, dtype=str, engine="python", on_bad_lines="skip")

        if "date" not in frame.columns and "system:index" in frame.columns:
            frame["date"] = (
                frame["system:index"].astype(str)
                .str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
                .str.replace("_", "-", regex=False)
            )
        if "date" not in frame.columns:
            raise ValueError(f"{path.name}: neither date nor system:index is available")

        frame["Region"] = region
        frame["Group"] = group
        frame["source_file"] = path.relative_to(PROJECT_ROOT).as_posix()
        frames.append(frame)

    if not frames:
        raise RuntimeError(f"No recognised multi-region NTL files were loaded from {folder.relative_to(PROJECT_ROOT)}")

    panel = pd.concat(frames, ignore_index=True)
    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize()
    panel = panel[panel["date"].between(DATE_START, DATE_END)].copy()
    panel["Valid_px"] = pd.to_numeric(panel["Valid_px"], errors="coerce")
    panel["Mean_by_sum"] = pd.to_numeric(panel["Mean_by_sum"], errors="coerce")
    return panel.sort_values(["Region", "Group", "date"]).reset_index(drop=True)


def mask_frame(group_frame):
    data = group_frame.copy().drop_duplicates("date").set_index("date").sort_index()
    valid = data["Valid_px"]
    raw = data["Mean_by_sum"].where(valid > 0)
    denominator = valid.loc[raw.notna()].max()
    if not np.isfinite(denominator) or denominator <= 0:
        raise ValueError("No valid direct observations for this region–mask combination")
    p95 = raw.quantile(0.95)
    return pd.DataFrame({
        "raw": raw,
        "ntl": raw.clip(upper=p95),
        "SC": 100 * valid / denominator,
        "Valid_px": valid,
    }).reindex(VALIDATION_DATES)


def load_ngcp_grid_1am_flat(xlsx_path, sheet_map):
    series_by_region = {}
    selected_columns = {}
    plot_series_by_region = {}
    for region, sheet_name in sheet_map.items():
        frame = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=1)
        frame.columns = [str(column).strip() for column in frame.columns]
        date_col = frame.columns[0]
        frame = frame.rename(columns={date_col: "DATE"})
        plot_dates = pd.to_datetime(frame["DATE"], errors="coerce")
        plot_frame = frame.drop(columns="DATE").set_index(plot_dates)
        frame["DATE"] = plot_dates.dt.normalize()
        frame = frame.dropna(subset=["DATE"]).set_index("DATE")
        frame.columns = [str(column).strip() for column in frame.columns]

        possible = ["1", "01", "1.0", "Hour 1", "HOUR 1", "1AM", "1am"]
        hour_col = next((column for column in possible if column in frame.columns), None)
        if hour_col is None:
            numeric_like = [
                column for column in frame.columns
                if str(column).replace(".0", "").isdigit()
            ]
            hour_col = numeric_like[0] if numeric_like else None
        if hour_col is None and len(frame.columns):
            hour_col = frame.columns[0]
        if hour_col is None:
            raise KeyError(f"No usable 01:00 load column in sheet {sheet_name}")

        values = pd.to_numeric(frame[hour_col], errors="coerce")
        values = values.groupby(level=0).mean().reindex(VALIDATION_DATES)
        series_by_region[region] = values.rename(region)
        selected_columns[region] = str(hour_col)
        plot_values = pd.to_numeric(plot_frame[hour_col], errors="coerce")
        plot_series_by_region[region] = plot_values.loc[
            (plot_values.index >= DATE_START) & (plot_values.index <= DATE_END)
        ]

    return pd.DataFrame(series_by_region), selected_columns, pd.DataFrame(plot_series_by_region).sort_index()


NTL_PANEL = read_multiregion_ntl(NTL_FOLDER)
NGCP_PANEL, NGCP_COLUMNS, ngcp_df = load_ngcp_grid_1am_flat(NGCP_XLSX_PATH, SHEET_MAP)

DATA_REGISTRY = {}
registry_rows = []
for region in REGION_ORDER:
    masks = {}
    source_files = []
    for code in ORDERED_MASKS:
        group = GROUP_LOOKUP[code]["suffix"]
        subset = NTL_PANEL[
            NTL_PANEL["Region"].eq(region) & NTL_PANEL["Group"].eq(group)
        ].copy()
        if subset.empty:
            raise FileNotFoundError(
                f"{region}: GHSL group {group} ({code}) is absent from {NTL_FOLDER.relative_to(PROJECT_ROOT)}"
            )
        masks[code] = mask_frame(subset)
        source_files.extend(subset["source_file"].dropna().unique().tolist())

    load = NGCP_PANEL[region].copy()
    DATA_REGISTRY[region] = {"masks": masks, "load": load}
    registry_rows.append({
        "region": region,
        "ntl_files": len(set(source_files)),
        "ghsl_masks": len(masks),
        "ngcp_sheet": SHEET_MAP[region],
        "load_column": NGCP_COLUMNS[region],
        "load_dates": int(load.notna().sum()),
    })

REGISTRY_TABLE = pd.DataFrame(registry_rows)
display(REGISTRY_TABLE)
REGISTRY_TABLE.to_csv(TABLE_SUPP / "data_registry.csv", index=False)

In [ ]:
def paired_metrics(ntl, load, direct_mask, window_days):
    frame = pd.DataFrame({"ntl": ntl, "load": load}).dropna()
    n_possible = int(load.notna().sum())
    n_pairs = len(frame)
    r = np.nan
    if (
        n_pairs >= 3
        and frame.ntl.std(ddof=0) > 0
        and frame.load.std(ddof=0) > 0
    ):
        r = float(frame.ntl.corr(frame.load))
    direct_n = int((direct_mask & load.notna()).sum())
    return {
        "n_pairs": n_pairs,
        "N_pct": 100 * n_pairs / n_possible if n_possible else np.nan,
        "direct_n": direct_n,
        "direct_pct": 100 * direct_n / n_possible if n_possible else np.nan,
        "r": r,
        "score": (n_pairs / n_possible) * max(r, 0) ** 2 if n_possible and np.isfinite(r) else 0.0,
    }

def evaluate_region(region, windows=WINDOWS, sc_grid=SC_GRID):
    load = DATA_REGISTRY[region]["load"]
    rows = []
    for code in ORDERED_MASKS:
        data = DATA_REGISTRY[region]["masks"][code]
        for tau in sc_grid:
            direct_mask = data.ntl.notna() & data.SC.ge(tau)
            qualified = data.ntl.where(direct_mask)
            direct_counts = direct_mask.astype(int)
            for width in windows:
                ntl_mean = qualified.rolling(int(width), min_periods=1, center=True).mean()
                load_mean = load.rolling(int(width), min_periods=1, center=True).mean()
                count = direct_counts.rolling(int(width), min_periods=1, center=True).sum()
                metrics = paired_metrics(ntl_mean, load_mean, direct_mask, int(width))
                paired = pd.DataFrame({"ntl": ntl_mean, "load": load_mean, "count": count}).dropna()
                rows.append({
                    "region": region,
                    "mask": code,
                    "mask_label": GROUP_LOOKUP[code]["label"],
                    "tau": float(tau),
                    "window_days": int(width),
                    "mean_direct_days": float(paired["count"].mean()) if len(paired) else np.nan,
                    "mean_window_support_pct": (
                        100 * float(paired["count"].mean()) / width if len(paired) else np.nan
                    ),
                    **metrics,
                })
    return pd.DataFrame(rows)

## 2. Native-daily evaluation and ranking

Evaluate the eight masks over the exact threshold grid at $W=1$. The displayed top-five table uses the 5% grid. Each row records Pearson $r$, coverage $N\%$, paired days $n$, and $(N\%/100)\max(r,0)^2$.

In [ ]:
DAILY_BY_REGION = []
for region in REGION_ORDER:
    daily = evaluate_region(region, windows=[1], sc_grid=SC_GRID)
    DAILY_BY_REGION.append(daily)
DAILY_ALL = pd.concat(DAILY_BY_REGION, ignore_index=True)
DAILY_ALL.to_csv(TABLE_SUPP / "all_subgrids_daily_mask_sc_results.csv.gz", index=False, compression="gzip")
print(f"{len(DAILY_ALL):,} daily five-sub-grid configurations calculated")

assert np.allclose(DAILY_ALL.N_pct, DAILY_ALL.direct_pct, equal_nan=True), (
    "At W=1, paired-estimate and direct-date coverage should match"
)

In [ ]:
# ============================================================
# Main Table — Top five daily configurations per sub-grid
# Restricted to τ = 5, 10, ..., 95%
# ============================================================

DISPLAY_TAUS_5 = np.arange(5, 100, 5, dtype=float)

REGIONAL_DAILY_5PCT = (
    DAILY_ALL[
        DAILY_ALL.window_days.eq(1)
        & DAILY_ALL.tau.isin(DISPLAY_TAUS_5)
    ]
    .copy()
)

# Preserve the intended regional order.
REGIONAL_DAILY_5PCT["region"] = pd.Categorical(
    REGIONAL_DAILY_5PCT["region"],
    categories=REGION_ORDER,
    ordered=True,
)

# Rank using the revised daily composite score.
REGIONAL_DAILY_5PCT = (
    REGIONAL_DAILY_5PCT
    .sort_values(
        [
            "region",
            "score",
            "N_pct",
            "r",
            "tau",
            "mask",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True,
            True,
        ],
    )
)

TOP5_CONFIGS_BY_REGION = (
    REGIONAL_DAILY_5PCT
    .groupby(
        "region",
        observed=True,
        group_keys=False,
    )
    .head(5)
    .copy()
)

TOP5_CONFIGS_BY_REGION["rank"] = (
    TOP5_CONFIGS_BY_REGION
    .groupby(
        "region",
        observed=True,
    )
    .cumcount()
    + 1
)

TOP5_CONFIGS_BY_REGION = (
    TOP5_CONFIGS_BY_REGION[
        [
            "region",
            "rank",
            "mask",
            "mask_label",
            "tau",
            "r",
            "N_pct",
            "n_pairs",
            "score",
        ]
    ]
    .sort_values(
        ["region", "rank"]
    )
    .reset_index(drop=True)
)

TOP5_CONFIGS_DISPLAY = (
    TOP5_CONFIGS_BY_REGION
    .rename(
        columns={
            "region": "Sub-grid",
            "rank": "Rank",
            "mask": "GHSL mask",
            "mask_label": "Mask description",
            "tau": "SC τ (%)",
            "r": "Pearson r",
            "N_pct": "N (%)",
            "n_pairs": "Paired days",
            "score": "Composite score",
        }
    )
    .copy()
)

TOP5_CONFIGS_DISPLAY["SC τ (%)"] = (
    TOP5_CONFIGS_DISPLAY["SC τ (%)"]
    .round(0)
    .astype(int)
)

TOP5_CONFIGS_DISPLAY["Pearson r"] = (
    TOP5_CONFIGS_DISPLAY["Pearson r"]
    .round(3)
)

TOP5_CONFIGS_DISPLAY["N (%)"] = (
    TOP5_CONFIGS_DISPLAY["N (%)"]
    .round(1)
)

TOP5_CONFIGS_DISPLAY["Composite score"] = (
    TOP5_CONFIGS_DISPLAY["Composite score"]
    .round(4)
)

display(
    Markdown(
        "## Top five native-daily configurations "
        "per sub-grid (5% τ grid)"
    )
)

display(TOP5_CONFIGS_DISPLAY)

TOP5_CONFIGS_DISPLAY.to_csv(
    TABLE_MAIN
    / "section2_top5_daily_configurations_by_subgrid_5pct_tau.csv",
    index=False,
)

TOP5_CONFIGS_BY_REGION.to_csv(
    TABLE_SUPP
    / "section2_top5_daily_configurations_by_subgrid_5pct_tau_raw.csv",
    index=False,
)

## 3. Regional alignment and coverage

The trade-off panels show the component metrics before score-based selection. The supplementary heatmaps retain the threshold-by-mask structure for each sub-grid.

In [ ]:
# ============================================================
# Native-daily alignment–coverage trade-off
# W=1; τ=5, 10, ..., 95%
# ============================================================

TAUS_5 = np.arange(5, 100, 5)

DAILY_TRADEOFF = DAILY_ALL.loc[
    DAILY_ALL.window_days.eq(1)
    & DAILY_ALL.tau.isin(TAUS_5)
].copy()

finite = DAILY_TRADEOFF[["N_pct", "r"]].replace(
    [np.inf, -np.inf], np.nan
).dropna()

if finite.empty:
    raise ValueError("No finite native-daily trade-off results.")

xmax = min(100, max(10, np.ceil(finite.N_pct.max() / 10) * 10))
ymin = max(-1, np.floor(finite.r.min() * 10) / 10 - 0.05)
ymax = min(1, np.ceil(finite.r.max() * 10) / 10 + 0.08)

positions = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2)]

fig = make_subplots(
    rows=2,
    cols=3,
    specs=[[{}, {}, {}], [{}, {}, None]],
    subplot_titles=list(REGION_ORDER),
    horizontal_spacing=0.07,
    vertical_spacing=0.18,
)

for heading in fig.layout.annotations:
    heading.font = dict(family="Arial", size=23, color=INK)

selected_rows = []

for region, (row, col) in zip(REGION_ORDER, positions):
    regional = DAILY_TRADEOFF.loc[
        DAILY_TRADEOFF.region.eq(region)
    ]

    for code in ORDERED_MASKS:
        q = regional.loc[
            regional["mask"].eq(code)
        ].sort_values("tau")

        if q.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=q.N_pct,
                y=q.r,
                mode="lines+markers",
                legendgroup=code,
                showlegend=False,
                line=dict(color=MASK_COLORS[code], width=2.4),
                marker=dict(
                    color=MASK_COLORS[code],
                    size=10,
                    opacity=0.95,
                    line=dict(color=INK, width=0.7),
                ),
                connectgaps=False,
                customdata=np.column_stack([
                    q.tau, q.score, q.n_pairs,
                ]),
                hovertemplate=(
                    f"{region} · {code}<br>"
                    "Spatial completeness threshold, τ=%{customdata[0]:.0f}%<br>"
                    "Pearson r=%{y:.2f}<br>"
                    "Retained direct-date coverage, N=%{x:.1f}%<br>"
                    "Composite score=%{customdata[1]:.3f}<br>"
                    "Paired days, n=%{customdata[2]:.0f}"
                    "<extra></extra>"
                ),
            ),
            row=row, col=col,
        )

    candidates = regional.dropna(subset=["score", "N_pct", "r"])
    if candidates.empty:
        raise ValueError(f"No selectable configurations: {region}")

    # Preserve the existing selection and tie-breaking rules.
    best = candidates.sort_values(
        ["score", "N_pct", "r", "tau", "mask"],
        ascending=[False, False, False, True, True],
    ).iloc[0]
    selected_rows.append(best)

    fig.add_trace(
        go.Scatter(
            x=[best.N_pct],
            y=[best.r],
            mode="markers+text",
            showlegend=False,
            text=[f"τ={best.tau:.0f}%"],
            textposition="top center",
            textfont=dict(family="Arial", size=18, color=INK),
            marker=dict(
                symbol="star",
                size=24,
                color=MASK_COLORS[best["mask"]],
                line=dict(color="black", width=1.5),
            ),
            customdata=[[
                best["mask"], best["mask_label"],
                best["score"], best["n_pairs"],
            ]],
            hovertemplate=(
                f"{region}<br>"
                "%{customdata[0]}: %{customdata[1]}<br>"
                f"Spatial completeness threshold, τ={best.tau:.0f}%<br>"
                "Pearson r=%{y:.2f}<br>"
                "Retained direct-date coverage, N=%{x:.1f}%<br>"
                "Composite score=%{customdata[2]:.3f}<br>"
                "Paired days, n=%{customdata[3]:.0f}"
                "<extra>Selected configuration</extra>"
            ),
        ),
        row=row, col=col,
    )

    fig.update_xaxes(
        range=[0, min(100, xmax + 4)],
        title_text="Retained direct-date coverage, N (%)",
        title_font=dict(family="Arial", size=19, color=INK),
        tickfont=dict(family="Arial", size=18, color=INK),
        dtick=20,
        gridcolor=GRID,
        showline=True,
        mirror=True,
        linecolor=INK,
        linewidth=1.4,
        ticks="outside",
        ticklen=6,
        row=row, col=col,
    )
    fig.update_yaxes(
        range=[ymin, ymax],
        title_text="Pearson r" if col == 1 else None,
        title_font=dict(family="Arial", size=20, color=INK),
        tickfont=dict(family="Arial", size=18, color=INK),
        dtick=0.2,
        gridcolor=GRID,
        zeroline=True,
        zerolinecolor="#A9B9C9",
        showline=True,
        mirror=True,
        linecolor=INK,
        linewidth=1.4,
        ticks="outside",
        ticklen=6,
        row=row, col=col,
    )

# Shared legend in the unused lower-right position.
for code in ORDERED_MASKS:
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="lines+markers",
            name=f"{code}: {GROUP_LOOKUP[code]['label']}",
            legendgroup=code,
            line=dict(color=MASK_COLORS[code], width=4),
            marker=dict(
                color=MASK_COLORS[code],
                size=10,
                line=dict(color=INK, width=0.7),
            ),
            showlegend=True,
            hoverinfo="skip",
        ),
        row=1, col=1,
    )

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        name="Selected configuration",
        marker=dict(
            symbol="star",
            size=21,
            color="#B0B0B0",
            line=dict(color="black", width=1.3),
        ),
        showlegend=True,
        hoverinfo="skip",
    ),
    row=1, col=1,
)

fig.update_layout(
    template="plotly_white",
    width=1450,
    height=1100,
    margin=dict(l=90, r=35, t=65, b=85),
    font=dict(family="Arial", color=INK, size=18),
    legend=dict(
        x=0.73,
        y=0.42,
        xanchor="left",
        yanchor="top",
        title=dict(text="GHSL mask", font=dict(size=20)),
        font=dict(size=18),
        tracegroupgap=3,
        itemsizing="constant",
    ),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

SELECTED_DAILY_TRADEOFF = pd.DataFrame(selected_rows)
fig_tradeoff = fig
fig_tradeoff.show()



stem = "fig_native_daily_alignment_coverage"

export_figure(fig_tradeoff, FIG_MAIN / stem, width=1450, height=1100, scale=3)

In [ ]:
# ============================================================
# Figure 8 — Native-daily configuration sweep by sub-grid
# ============================================================

DISPLAY_TAUS_5 = np.arange(5, 100, 5)


def regional_daily_heatmaps():
    regions = ["Bohol", "Cebu", "Negros", "Panay"]
    daily = DAILY_ALL.loc[
        DAILY_ALL["window_days"].eq(1)
        & DAILY_ALL["tau"].isin(DISPLAY_TAUS_5)
    ].copy()

    regional = daily.loc[daily["region"].isin(regions)]
    selected = (
        regional.dropna(subset=["score"])
        .sort_values("score", ascending=False, kind="stable")
        .drop_duplicates("region")
        .set_index("region")
        .reindex(regions)
    )
    if selected["score"].isna().any():
        missing = selected.index[selected["score"].isna()].tolist()
        raise ValueError(f"No valid native-daily score for: {', '.join(missing)}")

    # Match Figure 6's score limits using its Samar–Leyte display grid.
    score_max = 0.13

    # score_max = float(daily.loc[daily["region"].eq("Samar-Leyte"), "score"].max())
    if not np.isfinite(score_max) or score_max <= 0:
        raise ValueError("Cannot determine the Figure 6 score limit from DAILY_ALL.")

    score_scale = [
        [0 / 6, "#fef6b5"],
        [1 / 6, "#ffdd9a"],
        [2 / 6, "#ffc285"],
        [3 / 6, "#ffa679"],
        [4 / 6, "#fa8a76"],
        [5 / 6, "#f16d7a"],
        [6 / 6, "#e15383"],
    ]
    score_ticks = np.linspace(0, score_max, 5).tolist()

    metrics = [
        ("r", "Pearson r", "Blues", 0.60,
         [0, 0.20, 0.40, 0.60], ["0", "0.20", "0.40", "≥0.60"], ".3f"),
        ("N_pct", "Retained coverage (%)", "Greens", 50,
         [0, 10, 20, 30, 40, 50], ["0", "10", "20", "30", "40", "≥50"], ".1f"),
        ("score", "Composite score", score_scale, score_max,
         score_ticks, [f"{value:.2f}" for value in score_ticks], ".4f"),
    ]

    fig = make_subplots(
        rows=3,
        cols=4,
        shared_xaxes="all",
        shared_yaxes="all",
        horizontal_spacing=0.030,
        vertical_spacing=0.060,
        column_titles=regions,
    )

    y_labels = [f"{mask}: {GROUP_LOOKUP[mask]['label']}" for mask in ORDERED_MASKS]
    y_positions = np.arange(len(ORDERED_MASKS))
    mask_position = dict(zip(ORDERED_MASKS, y_positions))

    for row, (metric, title, scale, cmax, ticks, tick_text, fmt) in enumerate(
        metrics, start=1
    ):
        coloraxis = "coloraxis" if row == 1 else f"coloraxis{row}"
        first_panel_yaxis = "yaxis" if row == 1 else f"yaxis{(row - 1) * 4 + 1}"
        y0, y1 = fig.layout[first_panel_yaxis].domain

        fig.update_layout({
            coloraxis: dict(
                colorscale=scale,
                cmin=0,
                cmax=cmax,
                colorbar=dict(
                    title=dict(text=title, side="right", font=dict(size=18)),
                    x=1.015,
                    y=(y0 + y1) / 2,
                    len=(y1 - y0) * 0.92,
                    thickness=17,
                    tickvals=ticks,
                    ticktext=tick_text,
                    tickfont=dict(size=15),
                    ticks="outside",
                    ticklen=4,
                    outlinewidth=0.7,
                    outlinecolor=INK,
                ),
            )
        })

        for col, region in enumerate(regions, start=1):
            subset = regional.loc[regional["region"].eq(region)]
            matrix = (
                subset.pivot(index="mask", columns="tau", values=metric)
                .reindex(index=ORDERED_MASKS, columns=DISPLAY_TAUS_5)
            )
            hover_masks = np.broadcast_to(
                np.asarray(y_labels, dtype=object)[:, None], matrix.shape
            )

            fig.add_trace(
                go.Heatmap(
                    x=DISPLAY_TAUS_5,
                    y=y_positions,
                    z=matrix.to_numpy(dtype=float),
                    customdata=hover_masks,
                    coloraxis=coloraxis,
                    xgap=1,
                    ygap=1,
                    hoverongaps=False,
                    hovertemplate=(
                        f"{region}<br>%{{customdata}}<br>"
                        "SC threshold τ=%{x:.0f}%<br>"
                        f"{title}: %{{z:{fmt}}}<extra></extra>"
                    ),
                ),
                row=row,
                col=col,
            )

            # Select once using S_daily and outline the same cell in every row.
            tau = float(selected.at[region, "tau"])
            y = mask_position[selected.at[region, "mask"]]
            fig.add_shape(
                type="rect",
                x0=tau - 2.25,
                x1=tau + 2.25,
                y0=y - 0.44,
                y1=y + 0.44,
                line=dict(color="#ff0000", width=3),
                fillcolor="rgba(0,0,0,0)",
                layer="above",
                row=row,
                col=col,
            )

            fig.update_xaxes(
                range=[2.5, 97.5],
                tickvals=[5, 20, 40, 60, 80, 95],
                showticklabels=(row == 3),
                tickfont=dict(size=16),
                ticks="outside" if row == 3 else "",
                ticklen=4,
                showgrid=False,
                zeroline=False,
                showline=True,
                mirror=True,
                linecolor=INK,
                linewidth=1,
                row=row,
                col=col,
            )
            fig.update_yaxes(
                range=[-0.5, len(ORDERED_MASKS) - 0.5],
                tickvals=y_positions,
                ticktext=y_labels,
                showticklabels=(col == 1),
                tickfont=dict(size=16),
                ticks="",
                showgrid=False,
                zeroline=False,
                showline=True,
                mirror=True,
                linecolor=INK,
                linewidth=1,
                row=row,
                col=col,
            )

    for heading in fig.layout.annotations:
        heading.font = dict(family="Arial", size=21, color=INK)

    fig.add_annotation(
        x=0.5,
        y=-0.065,
        xref="paper",
        yref="paper",
        xanchor="center",
        yanchor="top",
        text="Spatial completeness threshold, τ (%)",
        showarrow=False,
        font=dict(family="Arial", size=19, color=INK),
    )

    width, height = 1640, 890
    fig.update_layout(
        template="plotly_white",
        width=width,
        height=height,
        margin=dict(l=215, r=145, t=60, b=105),
        font=dict(family="Arial", size=16, color=INK),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )

    fig.show()
    return export_figure(fig, (FIG_SUPP) / ("fig_08_regional_daily_heatmaps"), width=width, height=height)


FIG_08_REGIONAL_DAILY = regional_daily_heatmaps()

In [ ]:
# ============================================================
# Regional native-daily heatmaps
# Style matched to Notebook 13
# ============================================================

DISPLAY_TAUS_5 = np.arange(5, 100, 5, dtype=float)


def regional_mask_tau_matrix(
    frame,
    region,
    metric,
    display_taus=DISPLAY_TAUS_5,
):
    subset = frame[
        frame.region.eq(region)
        & frame.window_days.eq(1)
    ]

    return (
        subset
        .pivot(index="mask", columns="tau", values=metric)
        .reindex(index=ORDERED_MASKS, columns=display_taus)
    )


def plot_regional_daily_heatmap(frame, region, metric):
    matrix = regional_mask_tau_matrix(
        frame=frame,
        region=region,
        metric=metric,
    )

    z = matrix.to_numpy(dtype=float)

    y_labels = [
        f"{code}: {GROUP_LOOKUP[code]['label']}"
        for code in matrix.index
    ]

    if metric == "r":
        colorscale = "Blues"
        zmin, zmax = 0, 0.60
        highlight = z >= 0.60
        text_format = "{:.2f}"

        colorbar = dict(
            title=dict(text="Pearson r", side="right"),
            len=0.85,
            tickvals=[0, 0.20, 0.40, 0.60],
            ticktext=["0", "0.20", "0.40", "≥0.60"],
        )

        metric_name = "Pearson r"
        file_tag = "pearson_r"

    elif metric == "N_pct":
        colorscale = "Greens"
        zmin, zmax = 0, 50
        highlight = z >= 50
        text_format = "{:.0f}"

        colorbar = dict(
            title=dict(text="N (%)", side="right"),
            len=0.85,
            tickvals=[0, 10, 20, 30, 40, 50],
            ticktext=["0", "10", "20", "30", "40", "≥50"],
        )

        metric_name = "Direct-date N%"
        file_tag = "direct_N"

    elif metric == "score":
        colorscale = "Pinkyl"
        zmin = 0

        finite_scores = z[np.isfinite(z)]

        zmax = (
            max(0.01, float(finite_scores.max()))
            if finite_scores.size
            else 0.01
        )

        regional_top_decile = (
            np.percentile(finite_scores, 90)
            if finite_scores.size
            else np.nan
        )

        highlight = z >= regional_top_decile
        text_format = "{:.2f}"

        score_ticks = np.linspace(zmin, zmax, 5)

        colorbar = dict(
            title=dict(text="Composite score", side="right"),
            len=0.85,
            tickvals=score_ticks,
            ticktext=[f"{value:.2f}" for value in score_ticks],
        )

        metric_name = "Composite score"
        file_tag = "composite_score"

    else:
        raise ValueError(
            "metric must be 'r', 'N_pct', or 'score'"
        )

    fig = go.Figure()

    fig.add_trace(
        go.Heatmap(
            z=z,
            x=matrix.columns,
            y=y_labels,
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            xgap=2,
            ygap=2,
            colorbar=colorbar,
            showscale=True,
            hovertemplate=(
                f"{region}<br>"
                "Mask %{y}<br>"
                "Spatial completeness τ=%{x:.0f}%<br>"
                f"{metric_name}=%{{z:.3f}}"
                "<extra></extra>"
            ),
        )
    )

    # Annotate finite cells; highlighted values are bold and white.
    for row in range(z.shape[0]):
        for col in range(z.shape[1]):
            value = z[row, col]

            if not np.isfinite(value):
                continue

            value_text = text_format.format(value)

            if highlight[row, col]:
                value_text = f"<b>{value_text}</b>"

            fig.add_annotation(
                x=matrix.columns[col],
                y=y_labels[row],
                text=value_text,
                showarrow=False,
                font=dict(
                    family="Arial",
                    size=14,
                    color=(
                        "white"
                        if highlight[row, col]
                        else "black"
                    ),
                ),
            )

    fig.update_layout(
        template="plotly_white",
        width=1500,
        height=400,
        title=None,
        margin=dict(
            l=175,
            r=45,
            t=20,
            b=80,
        ),
        xaxis=dict(
            title="Spatial completeness threshold, τ (%)",
            tickvals=list(range(0, 101, 20)),
            tickfont=dict(size=14),
        ),
        yaxis=dict(
            tickfont=dict(size=14),
        ),
        font=dict(
            family="Arial",
            color=INK,
            size=16,
        ),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    safe_region = (
        region.lower()
        .replace("–", "_")
        .replace("-", "_")
        .replace(" ", "_")
    )

    stem = (
        f"section2_{safe_region}_"
        f"native_daily_{file_tag}"
    )

    return fig, stem


# ============================================================
# Generate and export all regional panels
# ============================================================

ALL_REGIONAL_HEATMAPS = {}

for region in REGION_ORDER:
    display(
        Markdown(
            f"## {region}"
        )
    )

    fig_r, stem_r = plot_regional_daily_heatmap(
        frame=DAILY_ALL,
        region=region,
        metric="r",
    )

    fig_N, stem_N = plot_regional_daily_heatmap(
        frame=DAILY_ALL,
        region=region,
        metric="N_pct",
    )

    fig_score, stem_score = plot_regional_daily_heatmap(
        frame=DAILY_ALL,
        region=region,
        metric="score",
    )

    fig_r.show()
    fig_N.show()
    fig_score.show()

    saved_r = export_figure(
        fig_r,
        FIG_SUPP / stem_r,
        width=1500,
        height=400,
    )

    saved_N = export_figure(
        fig_N,
        FIG_SUPP / stem_N,
        width=1500,
        height=400,
    )

    saved_score = export_figure(
        fig_score,
        FIG_SUPP / stem_score,
        width=1500,
        height=400,
    )

    ALL_REGIONAL_HEATMAPS[region] = {
        "r": saved_r,
        "N_pct": saved_N,
        "score": saved_score,
    }

## 4. Figure 8 — Selected regional time series and Rai zoom

The four sub-grid panels show the selected configurations recorded in the analysis. The full record and Rai zoom use the same normalisation. Native-daily points and trailing 30-day curves share the existing publication layout.

In [ ]:
# ----------------------------------------------------------
# Project and input paths
# ----------------------------------------------------------
PROJECT_ROOT = ROOT

DATE_START = pd.Timestamp(globals().get("DATE_START", "2012-01-19"))
DATE_END = pd.Timestamp(globals().get("DATE_END", "2024-12-31"))
ANALYSIS_DAYS = (DATE_END - DATE_START).days + 1
VALUE_COL = globals().get("VALUE_COL", "Mean_by_sum")

# Preserve metadata/helpers already defined in the notebook.
if "GHSL_META" not in globals():
    GHSL_META = {
        "30": {"code": "G1", "label": "Urban Core", "color": "#ff0000"},
        "23-30": {"code": "G2", "label": "Dense Urban", "color": "#732600"},
        "22-30": {"code": "G3", "label": "Core & Inner", "color": "#a87000"},
        "21-30": {"code": "G4", "label": "Urban Fringe", "color": "#ffff00"},
        "13-30": {"code": "G5", "label": "Built-Up / Rural Mix", "color": "#375623"},
        "12-30": {"code": "G6", "label": "Low–Mid Rural", "color": "#abcd66"},
        "11-30": {"code": "G7", "label": "Rural Settlements", "color": "#cdf57a"},
        "10-30": {"code": "G8", "label": "All Land", "color": "#7ab6f5"},
    }

if "GHSL_ORDER" not in globals():
    GHSL_ORDER = [
        "30", "23-30", "22-30", "21-30",
        "13-30", "12-30", "11-30", "10-30",
    ]

if "STORM_MARKERS" not in globals():
    STORM_MARKERS = [
        ("Haiyan", "2013-11-08"),
        ("Rai", "2021-12-16"),
    ]

if "zscore" not in globals():
    def zscore(series):
        s = pd.Series(series).astype(float)
        sd = s.std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            return pd.Series(np.nan, index=s.index)
        return (s - s.mean()) / sd

df_all = NTL_PANEL

In [ ]:
SETTINGS = [
    ("Bohol",  "30",    40, (-3, 3)),
    ("Cebu",   "30",    50, (-3, 3)),
    ("Negros", "30",    15, None),
    ("Panay",  "21-30",  5, None),
]

dates = pd.date_range(DATE_START, DATE_END, freq="D")
event = pd.Timestamp("2021-12-16")
zoom_start = event - pd.Timedelta(days=60)
zoom_end = event + pd.Timedelta(days=180)

ngcp_outlier_tables = {}


def prepare_selected_setting(region, group, tau):
    # Screen each region's NGCP series once.
    ng = pd.to_numeric(ngcp_df[region], errors="coerce").copy()
    ng.index = pd.to_datetime(ng.index).normalize()
    if ng.index.has_duplicates:
        raise ValueError(f"Duplicate NGCP dates: {region}")
    ng = ng.sort_index().reindex(dates)

    window = ng.rolling(31, center=True, min_periods=7)
    median = window.median()
    mad = window.apply(
        lambda x: np.nanmedian(np.abs(x - np.nanmedian(x))),
        raw=True,
    )
    sigma = 1.4826022185 * mad
    flagged = (sigma > 0) & ((ng - median).abs() > 3 * sigma)

    ngcp_outlier_tables[region] = pd.DataFrame({
        "NGCP_MW": ng,
        "Lower_limit_MW": median - 3 * sigma,
        "Upper_limit_MW": median + 3 * sigma,
    }).loc[flagged]
    ng = ng.mask(flagged)

    # Select only the columns needed for this configuration.
    gdf = df_all.loc[
        df_all["Region"].eq(region)
        & df_all["Group"].eq(group)
        & df_all["Valid_px"].gt(0),
        ["date", VALUE_COL, "Valid_px"],
    ].copy()

    if gdf.empty:
        raise ValueError(f"No NTL observations: {region}, {group}")

    hi = np.nanpercentile(gdf[VALUE_COL], 95)
    gdf = gdf.loc[gdf[VALUE_COL].le(hi)].copy()
    if gdf.empty:
        raise ValueError(f"No NTL observations after filtering: {region}")
    if gdf["date"].duplicated().any():
        raise ValueError(f"Duplicate NTL dates: {region}, {group}")

    gdf["SC_pct"] = gdf["Valid_px"] / gdf["Valid_px"].max() * 100
    gdf = gdf.set_index("date").sort_index().reindex(dates)
    ntl = gdf[VALUE_COL].where(gdf["SC_pct"] > tau)

    paired = ntl.notna() & ng.notna()
    reference = ntl.loc[paired]
    sd = reference.std(ddof=0)

    ntl_scaled = (
        (ntl - reference.mean()) / sd
        if np.isfinite(sd) and sd > 0
        else pd.Series(np.nan, index=dates)
    )
    ng_daily = zscore(ng)

    # Smoothed curves are daily estimates from available observations.
    # Windows with no observations remain missing.
    return pd.DataFrame({
        "ntl1": ntl_scaled.where(paired),
        "ntl30": ntl_scaled.rolling("30D", min_periods=1).mean(),
        "ng1": ng_daily,
        "ng30": ng_daily.rolling("30D", min_periods=1).mean(),
    })


# Prepare each configuration once; reuse it for both columns.
prepared = {
    (region, group, tau): prepare_selected_setting(region, group, tau)
    for region, group, tau, _ in SETTINGS
}

HEIGHT, TOP, BOTTOM = 1250, 75, 85
inner_height = HEIGHT - TOP - BOTTOM

fig = make_subplots(
    rows=len(SETTINGS),
    cols=2,
    shared_xaxes="columns",
    column_widths=[0.75, 0.25],
    horizontal_spacing=0.045,
    vertical_spacing=0.035,
)

views = [
    (pd.Timestamp(DATE_START), pd.Timestamp(DATE_END)),
    (zoom_start, zoom_end),
]

for row, (region, group, tau, fixed_limits) in enumerate(SETTINGS, 1):
    data = prepared[(region, group, tau)]
    meta = GHSL_META[group]

    if fixed_limits is not None:
        limits = list(fixed_limits)
        ticks = [-3, 0, 3]
    else:
        values = data.to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if not values.size:
            raise ValueError(f"No finite comparison data: {region}")
        padding = max(0.04 * np.ptp(values), 0.1)
        limits = [
            min(values.min() - padding, -2.2),
            max(values.max() + padding, 2.2),
        ]
        ticks = [-2, 0, 2]

    base_bottom, base_top = fig.get_subplot(row, 1).yaxis.domain
    plot_top = base_top - 38 / inner_height

    for col, (start, end) in enumerate(views, 1):
        view = data.loc[start:end]
        x = view.index.strftime("%Y-%m-%d").tolist()

        paired = view[["ntl1", "ng1"]].notna().all(axis=1)
        count = int(paired.sum())
        possible_days = (
            ANALYSIS_DAYS if col == 1 else (end - start).days + 1
        )
        coverage = 100 * count / possible_days

        if count >= 2:
            r1 = view.loc[paired, "ntl1"].corr(
                view.loc[paired, "ng1"]
            )
            r30 = view.loc[paired, "ntl30"].corr(
                view.loc[paired, "ng30"]
            )
        else:
            r1 = r30 = np.nan

        r1_text = f"{r1:.2f}" if np.isfinite(r1) else "—"
        r30_text = f"{r30:.2f}" if np.isfinite(r30) else "—"

        for key, is_ngcp, mode in [
            ("ng1", True, "markers"),
            ("ntl1", False, "markers"),
            ("ntl30", False, "lines"),
            ("ng30", True, "lines"),
        ]:
            color = (
                ("#666666" if mode == "markers" else "black")
                if is_ngcp else meta["color"]
            )

            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=view[key].to_numpy(),
                    mode=mode,
                    marker=dict(
                        color=color,
                        size=3.2 if col == 1 else 4.8,
                        opacity=(
                            (0.13 if col == 1 else 0.22)
                            if is_ngcp else
                            (0.24 if col == 1 else 0.38)
                        ),
                    ),
                    line=dict(
                        color=color,
                        width=(
                            (1.9 if col == 1 else 2.5)
                            if is_ngcp else
                            (1.7 if col == 1 else 2.2)
                        ),
                    ),
                    connectgaps=False,
                    showlegend=False,
                    hovertemplate="%{x}<br>z=%{y:.2f}<extra></extra>",
                ),
                row=row, col=col,
            )

        label = (
            f"{region} · {meta['code']}: {meta['label']} · τ={tau}%"
            if col == 1 else meta["code"]
        )
        dot = f"<span style='color:{meta['color']}'>●</span>"
        line = f"<span style='color:{meta['color']}'>━</span>"

        fig.add_annotation(
            x=fig.get_subplot(row, col).xaxis.domain[1],
            y=base_top,
            xref="paper",
            yref="paper",
            text=(
                f"{label}  {dot} r₁={r1_text}  "
                f"{line} r₃₀={r30_text}  "
                f"N={coverage:.0f}%  n={count}"
            ),
            xanchor="right",
            yanchor="top",
            showarrow=False,
            font=dict(size=18 if col == 1 else 17),
        )

        fig.update_yaxes(
            domain=[base_bottom, plot_top],
            range=limits,
            tickmode="array",
            tickvals=ticks,
            ticktext=[
                f"−{abs(v)}" if v < 0 else f"+{v}" if v > 0 else "0"
                for v in ticks
            ],
            tickfont=dict(size=17),
            showticklabels=True,
            zeroline=True,
            zerolinecolor="#dddddd",
            row=row, col=col,
        )
        fig.update_xaxes(
            type="date",
            range=[
                start.strftime("%Y-%m-%d"),
                end.strftime("%Y-%m-%d"),
            ],
            showticklabels=False,
            showgrid=False,
            row=row, col=col,
        )

        if col == 1:
            fig.update_yaxes(
                title_text="z-score",
                title_font=dict(size=19),
                row=row, col=col,
            )
            fig.add_vrect(
                x0=zoom_start.strftime("%Y-%m-%d"),
                x1=zoom_end.strftime("%Y-%m-%d"),
                line_color="#555555",
                line_width=1.5,
                fillcolor="rgba(0,0,0,0)",
                layer="above",
                row=row, col=col,
            )
        else:
            for update_axis in (fig.update_xaxes, fig.update_yaxes):
                update_axis(
                    showline=True,
                    mirror=True,
                    linecolor="#555555",
                    linewidth=1.5,
                    row=row, col=col,
                )

        for _, event_date in STORM_MARKERS:
            if start <= pd.Timestamp(event_date) <= end:
                fig.add_vline(
                    x=event_date,
                    line_color="black",
                    line_dash="dash",
                    line_width=0.9,
                    opacity=0.5,
                    row=row, col=col,
                )

# Preserve your refined shared-key placement.
fig.add_annotation(
    x=0.6,
    y=0.82,
    xref="paper",
    yref="paper",
    text=(
        "NGCP Electric Load<br>"
        "<span style='color:#666666'>●</span> W=1 "
        "<span style='color:black'>━</span> W=30"
    ),
    xanchor="left",
    yanchor="top",
    align="left",
    showarrow=False,
    font=dict(family="Arial", size=16, color="#333333"),
)

for col, tickformat, dtick in [
    (1, "%Y", "M48"),
    (2, "%b %Y", "M3"),
]:
    fig.update_xaxes(
        showticklabels=True,
        title_text="Date",
        title_font=dict(size=19),
        tickformat=tickformat,
        dtick=dtick,
        tickfont=dict(size=17),
        tickangle=0,
        row=len(SETTINGS), col=col,
    )

fig.update_layout(
    template="plotly_white",
    width=1550,
    height=HEIGHT,
    margin=dict(t=TOP, b=BOTTOM, l=95, r=45),
    font=dict(family="Arial", size=18, color="#333333"),
    paper_bgcolor="rgba(0,0,0,0)",
    showlegend=False,
)

fig_selected_regions_rai = fig
fig_selected_regions_rai.show()

In [ ]:
for figure, filename in [
    (fig_selected_regions_rai, "fig_08_selected_regions_rai"),
]:
    export_figure(
        figure,
        FIG_MAIN / filename,
        width=figure.layout.width,
        height=figure.layout.height,
        scale=3,
    )

## 5. Additional regional diagnostics

Set RUN_ADDITIONAL_DIAGNOSTICS=True in setup to reproduce the earlier regional NTL viewers and load context. These export to the diagnostic folders.

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    import json
    import plotly.io as pio

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    PROJECT_ROOT = ROOT

    DATASET_DIR = PROJECT_ROOT / "datasets"

    OUTPUT_MAIN_FIG_DIR = FIG_MAIN
    FIG_DIAGNOSTIC = FIG_DIAGNOSTIC
    OUTPUT_MAIN_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "main"
    TABLE_DIAGNOSTIC = TABLE_DIAGNOSTIC

    for out_dir in [
        OUTPUT_MAIN_FIG_DIR,
        FIG_DIAGNOSTIC,
        OUTPUT_MAIN_TABLE_DIR,
        TABLE_DIAGNOSTIC,
    ]:
        out_dir.mkdir(parents=True, exist_ok=True)

    print(f"Main figures: {OUTPUT_MAIN_FIG_DIR.relative_to(PROJECT_ROOT)}")
    print(f"Supplementary figures: {FIG_DIAGNOSTIC.relative_to(PROJECT_ROOT)}")
    print(f"Main tables: {OUTPUT_MAIN_TABLE_DIR.relative_to(PROJECT_ROOT)}")
    print(f"Supplementary tables: {TABLE_DIAGNOSTIC.relative_to(PROJECT_ROOT)}")

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    ROLLING_WINDOW = 30

    REGION_COLORS = {
        "Bohol": "#4E79A7",
        "Cebu": "#D65A4A",
        "Negros": "#5AB87A",
        "Panay": "#8E66C6",
        "Samar-Leyte": "#F1A340",
    }

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    # ==============================
    # Shared Helpers
    # ==============================

    def date_str_series(x):
        return pd.to_datetime(x).dt.strftime("%Y-%m-%d")


    def add_event_markers(fig, y=1.04):
        for name, event_date in STORM_MARKERS:
            fig.add_vline(
                x=event_date,
                line_dash="dash",
                line_color="black",
                line_width=2,
            )

            fig.add_annotation(
                x=event_date,
                y=y,
                yref="paper",
                text=name,
                showarrow=False,
                font=dict(size=14, color="black"),
                bgcolor="white",
                bordercolor="black",
                borderwidth=1,
                xanchor="left",
            )

        return fig

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    # ==============================
    # Region-Specific NTL Time Series
    # ==============================

    def plot_region_ntl_timeseries(df_all, region, tau=40, w=30, value_col=VALUE_COL):
        df_region = df_all[df_all["Region"] == region].copy()

        if df_region.empty:
            raise ValueError(f"No NTL data found for region: {region}")

        groups = [g for g in GHSL_ORDER if g in df_region["Group"].unique()]

        fig = make_subplots(
            rows=2,
            cols=1,
            row_heights=[0.12, 0.78],
            shared_xaxes=True,
            vertical_spacing=0.08,
            subplot_titles=("Daily spatial completeness, SC (%)", "Mean DNB-BRDF radiance"),
        )

        buttons = []
        n_traces_per_group = 3

        for i, group in enumerate(groups):
            gdf = df_region[df_region["Group"] == group].copy()
            gdf = gdf[gdf["Valid_px"] > 0].copy().sort_values("date")

            if gdf.empty:
                continue

            hi = np.nanpercentile(gdf[value_col], 95)
            gdf = gdf[gdf[value_col] <= hi].copy()

            total_px = gdf["Valid_px"].max()
            gdf["SC_pct"] = gdf["Valid_px"] / total_px * 100
            gdf["date_plot"] = date_str_series(gdf["date"])

            fig.add_trace(
                go.Heatmap(
                    z=np.expand_dims(gdf["SC_pct"].values, axis=0),
                    x=gdf["date_plot"],
                    y=[0],
                    colorscale="Greens",
                    zmin=0,
                    zmax=100,
                    showscale=True,
                    colorbar=dict(title="SC (%)", len=0.25, thickness=18, x=0.95, y=0.95),
                    visible=(i == 0),
                ),
                row=1,
                col=1,
            )

            gdf = gdf[gdf["SC_pct"] > tau].copy()

            if not gdf.empty:
                gdf[f"MA_{w}d"] = gdf[value_col].rolling(w, min_periods=1).mean()
                gdf["date_plot"] = date_str_series(gdf["date"])

            group_color = GHSL_META.get(group, {}).get("color", "#999999")
            group_code = GHSL_META.get(group, {}).get("code", group)

            fig.add_trace(
                go.Scatter(
                    x=gdf["date_plot"],
                    y=gdf[value_col],
                    mode="markers",
                    marker=dict(color=group_color, size=4, opacity=0.18),
                    name=f"{group_code} daily",
                    visible=(i == 0),
                    showlegend=False,
                ),
                row=2,
                col=1,
            )

            fig.add_trace(
                go.Scatter(
                    x=gdf["date_plot"],
                    y=gdf[f"MA_{w}d"],
                    mode="lines",
                    line=dict(color=group_color, width=2.2),
                    name=f"{group_code} {w}d MA",
                    visible=(i == 0),
                ),
                row=2,
                col=1,
            )

            visibility = [False] * (n_traces_per_group * len(groups))
            for j in range(n_traces_per_group):
                visibility[i * n_traces_per_group + j] = True

            buttons.append(
                dict(
                    label=group_code,
                    method="update",
                    args=[
                        {"visible": visibility},
                        {"title": f"{region} DNB-BRDF NTL — {group_code} (SC > {tau}%)"},
                    ],
                )
            )

        add_event_markers(fig, y=0.86)

        fig.update_layout(
            updatemenus=[dict(type="buttons", buttons=buttons, direction="right", showactive=True, x=0.5, xanchor="center", y=-0.35, yanchor="top")],
            xaxis2=dict(title="", rangeslider=dict(visible=True), type="date"),
            yaxis=dict(showticklabels=False),
            yaxis2=dict(title="Radiance (nW·cm⁻²·sr⁻¹)"),
            template="plotly_white",
            paper_bgcolor="rgba(0,0,0,0)",
            legend=dict(orientation="h", y=0.88, x=0.8, font=dict(size=16)),
            height=650,
            width=1200,
            margin=dict(t=70, l=70, b=110, r=40),
        )

        fig.update_xaxes(tickfont=dict(size=14))
        fig.update_yaxes(tickfont=dict(size=14))

        return fig

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    fig_sx_region_ntl = plot_region_ntl_timeseries(df_all, region="Samar-Leyte", tau=40, w=ROLLING_WINDOW)
    fig_sx_region_ntl.show()

    export_figure(fig_sx_region_ntl, Path(FIG_DIAGNOSTIC / "fig_sx_region_ntl_timeseries_samar_leyte.html").with_suffix(""), static=True)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    # ==============================
    # Regional Comparison by GHSL Group
    # ==============================

    def plot_ma_by_class_group(df_all, tau=30, w=30, value_col=VALUE_COL):
        groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]
        regions = [r for r in REGION_ORDER if r in df_all["Region"].unique()]

        fig = go.Figure()
        buttons = []
        n_traces_per_group = len(regions)

        for i, group in enumerate(groups):
            gcode = GHSL_META.get(group, {}).get("code", group)
            gdesc = GHSL_META.get(group, {}).get("label", "")

            for region in regions:
                gdf = df_all[(df_all["Group"] == group) & (df_all["Region"] == region)].copy()
                gdf = gdf[gdf["Valid_px"] > 0].sort_values("date")

                if gdf.empty:
                    continue

                hi = np.nanpercentile(gdf[value_col], 95)
                gdf = gdf[gdf[value_col] <= hi].copy()

                total_px = gdf["Valid_px"].max()
                gdf["SC_pct"] = gdf["Valid_px"] / total_px * 100
                gdf = gdf[gdf["SC_pct"] > tau].copy()

                if gdf.empty:
                    continue

                gdf[f"MA_{w}d"] = gdf[value_col].rolling(w, min_periods=w).mean()
                gdf["date_plot"] = date_str_series(gdf["date"])

                fig.add_trace(
                    go.Scatter(
                        x=gdf["date_plot"],
                        y=gdf[f"MA_{w}d"],
                        mode="lines",
                        name=region,
                        line=dict(color=REGION_COLORS.get(region), width=2),
                        visible=(i == 0),
                    )
                )

            visibility = [False] * (n_traces_per_group * len(groups))
            for j in range(n_traces_per_group):
                visibility[i * n_traces_per_group + j] = True

            buttons.append(
                dict(
                    label=gcode,
                    method="update",
                    args=[{"visible": visibility}, {"title": f"{gcode}: {gdesc} — SC > {tau}%"}],
                )
            )

        add_event_markers(fig, y=0.88)

        fig.update_layout(
            xaxis=dict(title="", rangeslider=dict(visible=True), type="date"),
            yaxis=dict(title="Radiance (nW·cm⁻²·sr⁻¹)"),
            updatemenus=[dict(type="buttons", buttons=buttons, direction="right", showactive=True, x=0.5, xanchor="center", y=-0.35, yanchor="top")],
            legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center", font=dict(size=14)),
            template="plotly_white",
            height=600,
            width=1200,
            margin=dict(t=60, l=70, b=100, r=40),
            paper_bgcolor="rgba(0,0,0,0)",
        )

        return fig

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    fig_sx_region_by_group = plot_ma_by_class_group(df_all, tau=30, w=ROLLING_WINDOW)
    fig_sx_region_by_group.show()

    export_figure(fig_sx_region_by_group, Path(FIG_DIAGNOSTIC / "fig_sx_region_comparison_by_ghsl_group.html").with_suffix(""), static=True)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    ngcp_df.to_csv(
        TABLE_DIAGNOSTIC / "table_sx_ngcp_1am_load_visayas_2012_2024.csv",
        index=True,
    )

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    # ==============================
    # NGCP Combined 30-Day Moving Average
    # ==============================

    def plot_ngcp_combined_ma(df_grid, w=30):
        df = df_grid.copy()
        df.index = pd.to_datetime(df.index)
        df = df.sort_index()
        df = df[(df.index >= DATE_START) & (df.index <= DATE_END)]

        fig = go.Figure()

        for region in [r for r in REGION_ORDER if r in df.columns]:
            ma = df[region].rolling(window=w, min_periods=w).mean()

            fig.add_trace(
                go.Scatter(
                    x=[d.strftime("%Y-%m-%d") for d in ma.index],
                    y=ma,
                    mode="lines",
                    name=region,
                    line=dict(width=2.5, color=REGION_COLORS.get(region)),
                )
            )

        add_event_markers(fig, y=0.88)

        fig.update_layout(
            template="plotly_white",
            height=600,
            width=1200,
            xaxis=dict(title="Date", type="date", rangeslider=dict(visible=False)),
            yaxis=dict(title="Grid load (MW)"),
            legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center", font=dict(size=14)),
            margin=dict(t=80, l=70, b=80, r=40),
            paper_bgcolor="rgba(0,0,0,0)",
        )

        return fig

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    fig_sx_ngcp_load = plot_ngcp_combined_ma(ngcp_df, w=ROLLING_WINDOW)
    fig_sx_ngcp_load.show()

    export_figure(
        fig_sx_ngcp_load,
        FIG_DIAGNOSTIC / "fig_sx_ngcp_subgrid_load_ma30_2012_2024",
        width=1200,
        height=600,
        scale=3,
        force_date_axis=True,
        tickformat="%Y",
    )

## 6. Outputs

Figure 8 and the regional trade-off use outputs/figures/main/. Component surfaces use outputs/figures/supplementary/. The daily sweep read by Notebook 15 is in outputs/tables/supplementary/.

## Conclusion

Regional rankings describe the tested grids and their observation support. Interpret differences in selected masks and thresholds alongside r, N% and n. Notebook 15 examines how temporal averaging and score formulation alter those comparisons.